# 04 — Random Walk
**Week 2 | Mathematical Foundations for RL**

Random walks appear everywhere in RL — from the way an agent explores to the theoretical analysis
of TD learning. They also give great intuition about variance in stochastic processes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)

## 1. Simple 1D Random Walk
At each step: move +1 (right) or -1 (left) with equal probability.

In [ ]:
def random_walk_1d(n_steps, p_right=0.5):
    steps = np.where(np.random.rand(n_steps) < p_right, 1, -1)
    return np.concatenate([[0], np.cumsum(steps)])

n_steps = 500
fig, ax = plt.subplots(figsize=(10, 3.5))
for i in range(20):
    walk = random_walk_1d(n_steps)
    ax.plot(walk, alpha=0.4, linewidth=0.8)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Position')
ax.set_title('20 Random Walks (1D, 500 steps)')
plt.tight_layout(); plt.show()

## 2. Distribution of Positions at Time t
At time t, position X_t ~ N(0, t) — variance grows linearly with time.

In [ ]:
checkpoints = [10, 50, 100, 500]
n_walks = 10_000
fig, axes = plt.subplots(1, len(checkpoints), figsize=(14, 3), sharey=False)

for ax, t in zip(axes, checkpoints):
    positions = [random_walk_1d(t)[-1] for _ in range(n_walks)]
    ax.hist(positions, bins=40, color='steelblue', edgecolor='white', density=True)
    ax.set_title(f't = {t}\nstd ≈ {np.std(positions):.1f}')
    ax.set_xlabel('Position')

axes[0].set_ylabel('Density')
plt.suptitle('Distribution of position at time t', y=1.02)
plt.tight_layout(); plt.show()
print("Theoretical std = sqrt(t):", [f"{t}→{t**0.5:.1f}" for t in checkpoints])

## 3. Biased Random Walk
What if the agent has a preference? (p_right > 0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for p, color, label in [(0.5,'gray','p=0.5 (unbiased)'), (0.55,'steelblue','p=0.55'), (0.6,'seagreen','p=0.60')]:
    walks = np.array([random_walk_1d(500, p) for _ in range(500)])
    mean_walk = walks.mean(axis=0)
    std_walk  = walks.std(axis=0)
    x = np.arange(501)
    ax.plot(mean_walk, color=color, linewidth=2, label=label)
    ax.fill_between(x, mean_walk-std_walk, mean_walk+std_walk, alpha=0.15, color=color)
ax.set_xlabel('Step'); ax.set_ylabel('Mean position ± std')
ax.set_title('Biased vs Unbiased Random Walk')
ax.legend(); plt.tight_layout(); plt.show()

## 4. First Passage Time
How long until the walk reaches position +10 for the first time?

In [ ]:
def first_passage_time(target=10, max_steps=5000):
    pos = 0
    for t in range(1, max_steps+1):
        pos += np.random.choice([-1, 1])
        if pos >= target:
            return t
    return max_steps  # didn't reach target

fpt = [first_passage_time(10) for _ in range(5000)]
plt.figure(figsize=(7, 3))
plt.hist(fpt, bins=60, color='darkorange', edgecolor='white')
plt.xlabel('Steps to reach position +10'); plt.ylabel('Count')
plt.title(f'First Passage Time Distribution (mean={np.mean(fpt):.0f})')
plt.tight_layout(); plt.show()

## ✅ Exercises
1. Modify the 1D walk to stop when it hits +20 or -20. What fraction of walks end at +20 vs -20?
2. Simulate a **2D random walk** (move up/down/left/right). Plot 5 trajectories on an x-y grid.
3. **Challenge**: implement the classic '5-state random walk' from Sutton & Barto Example 6.2. States A–E, terminal states at each end. Compute true state values analytically and verify empirically.

Solution 1- 
Because the underlying walk is completely unbiased p=0.5 and the boundary limits are equidistant from the origin 0, the split is exactly 50% at +20 and 50% at -20.

In [ ]:
def bounded_walk(target=20):
    pos = 0
    while abs(pos) < target:
        pos += np.random.choice([-1, 1])
    return pos

results = [bounded_walk() for _ in range(2000)]
pos_20 = results.count(20) / len(results)
neg_20 = results.count(-20) / len(results)
print(f"Ends at +20: {pos_20:.2%}, Ends at -20: {neg_20:.2%}")

Solution 2-

In [ ]:
def random_walk_2d(n_steps=500):
    x, y = [0], [0]
    for _ in range(n_steps):
        dx, dy = np.random.choice([(0,1), (0,-1), (1,0), (-1,0)])
        x.append(x[-1] + dx)
        y.append(y[-1] + dy)
    return x, y

plt.figure(figsize=(6, 6))
for i in range(5):
    wx, wy = random_walk_2d()
    plt.plot(wx, wy, label=f'Walk {i+1}', alpha=0.8)
plt.plot(0, 0, 'ro', label='Start')
plt.title('5 Independent 2D Random Walks')
plt.xlabel('X position'); plt.ylabel('Y position'); plt.legend(); plt.grid(True); plt.show()

Solution 3-
Let the states be numbered $1$ to $5$ (corresponding to $A$ to $E$). Terminal Left is $0$, and Terminal Right is $6$. Since transitions are uniform left or right, the true expected values change linearly across the states: $V(A)=\frac{1}{6}$, $V(B)=\frac{2}{6}$, $V(C)=\frac{3}{6}$, $V(D)=\frac{4}{6}$, $V(E)=\frac{5}{6}$.

In [ ]:
def sutton_barto_walk(start_state=3):
    state = start_state
    while 0 < state < 6:
        state += np.random.choice([-1, 1])
    return 1 if state == 6 else 0  

# Evaluate empirically
empirical_values = {}
for state in range(1, 6):
    returns = [sutton_barto_walk(state) for _ in range(10_000)]
    empirical_values[chr(64 + state)] = np.mean(returns) 

print("True Analytical Values: {'A': 0.17, 'B': 0.33, 'C': 0.50, 'D': 0.67, 'E': 0.83}")
print("Empirical Estimation:", {k: round(v, 2) for k, v in empirical_values.items()})